# MACHINE LEARNING - Melhoria Temporal

# 1. Importação e Configuração

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    ConfusionMatrixDisplay
)

from lightgbm import LGBMClassifier

sns.set_theme(style="whitegrid")

# Configurações
SEMENTE = 42
N_SPLITS = 5

# Limites de registros
MAX_AMOSTRA_CV = 200_000
MAX_AMOSTRA_TREINO = 500_000

# Caminho dos arquivos processados pelo notebook original
PASTA_SPLITS = "datasets/splits"

# 2. Carregamento dos Dados

- Os três arquivos possuem os atributos processados e a variável alvo `Is_laundering`. 

- Como os arquivos foram salvos com `index=False`, o Pandas recria índices sequenciais na leitura. 

- A ordem original das linhas é preservada, portanto, podemos utilizar essa ordem cronológica.

In [2]:
# Carregamento das divisões
treino = pd.read_csv(f"{PASTA_SPLITS}/treino.csv")
validacao = pd.read_csv(f"{PASTA_SPLITS}/validacao.csv")
teste = pd.read_csv(f"{PASTA_SPLITS}/teste.csv")

# Separação entre atributos e variável-alvo
X_train = treino.drop(columns=["Is_laundering"])
y_train = treino["Is_laundering"]

X_val = validacao.drop(columns=["Is_laundering"])
y_val = validacao["Is_laundering"]

X_test = teste.drop(columns=["Is_laundering"])
y_test = teste["Is_laundering"]

# Conferência das dimensões e distribuição das classes
resumo = pd.DataFrame({
    "Registros": [
        len(X_train),
        len(X_val),
        len(X_test)
    ],
    "Positivas": [
        int(y_train.sum()),
        int(y_val.sum()),
        int(y_test.sum())
    ],
    "Taxa positiva (%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ]
}, index=["Treinamento", "Validação", "Teste"])

display(resumo)

,Registros,Positivas,Taxa positiva (%)
Treinamento,6653396,6764,0.101662
Validação,1425728,1412,0.099037
Teste,1425728,1697,0.119027


# 3. Pré-Processamento

Como o pré-processador está dentro de uma Pipeline, o escalonamento e a codificação são ajustados novamente em cada fold, utilizando somente os registros de treinamento daquele fold. Isso evita vazamento de informações durante a validação cruzada.

In [3]:
# Características numéricas
colunas_numericas = [
    "Amount_log",
    "dia_mes",
    "hora_sin",
    "hora_cos",
    "semana_sin",
    "semana_cos",
    "operacao_internacional",
    "mudanca_moeda",
    "envios_anteriores_log",
    "recebimentos_anteriores_log",
    "pares_anteriores_log",
    "razao_valor_media_anterior_log",
    "seg_desde_ultimo_envio_log",
]

# Características categóricas
colunas_categoricas = [
    "Payment_type",
    "Payment_currency",
    "Received_currency",
    "Sender_bank_location",
    "Receiver_bank_location",
]

# Escalonadores
ESCALONADORES = {
    "padrao": StandardScaler,
    "minmax": MinMaxScaler
}

# Construção do pré-processador
def construir_preprocessador(escalonador="padrao"):

    if escalonador not in ESCALONADORES:
        raise ValueError(
            f"Escalonador inválido: {escalonador}"
        )

    preprocessador = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputador", SimpleImputer(strategy="median")),
                    ("escalonador", ESCALONADORES[escalonador]())
                ]),
                colunas_numericas
            ),
            (
                "cat",
                Pipeline([
                    ("encoder", OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    ))
                ]),
                colunas_categoricas
            )
        ]
    )

    preprocessador.set_output(transform="pandas")

    return preprocessador

# 4. Modelos

O PCA não foi incluído nessa versão, pois o objetivo é isolar as duas mudanças na avaliação temporal, mantendo a configuração padrão dos modelos originais.

In [4]:
def criar_modelos(escalonador="padrao", semente=SEMENTE):

    return {

        "Dummy": DummyClassifier(
            strategy="most_frequent"
        ),

        "LogisticRegression": Pipeline([
            ("preprocessador", construir_preprocessador(escalonador)),
            ("classificador", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=semente
            ))
        ]),

        "RandomForest": Pipeline([
            ("preprocessador", construir_preprocessador(escalonador)),
            ("classificador", RandomForestClassifier(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=semente,
                n_jobs=-1
            ))
        ]),

        "LightGBM": Pipeline([
            ("preprocessador", construir_preprocessador(escalonador)),
            ("classificador", LGBMClassifier(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                subsample=0.8,
                subsample_freq=1,
                colsample_bytree=0.8,
                class_weight="balanced",
                importance_type="gain",
                random_state=semente,
                n_jobs=-1,
                verbose=-1
            ))
        ])
    }


modelos = criar_modelos("padrao")

list(modelos)

['Dummy', 'LogisticRegression', 'RandomForest', 'LightGBM']

# 5. Preparação da Amostra Temporal

- Selecionamos um intervalo contínuo do conjunto de treinamento.

- Foram escolhidos os últimos 200 mil registros para manter o limite de processamento.

- A validação cruzada analisará um intervalo contínuo do período de treinamento, sem misturar registros aleatórios.

In [5]:
# Seleciona os registros cronologicamente mais recentes
# dentro do conjunto de treinamento.

X_cv = X_train.iloc[-MAX_AMOSTRA_CV:].copy()
y_cv = y_train.loc[X_cv.index].copy()

# Verifica a ordenação
assert X_cv.index.is_monotonic_increasing

print("Amostra temporal:", X_cv.shape)
print("Transações positivas:", int(y_cv.sum()))
print(f"Taxa positiva: {y_cv.mean():.4%}")

Amostra temporal: (200000, 18)
Transações positivas: 227
Taxa positiva: 0.1135%


# 6. Validação Cruzada com TimeSeriesSplit

### 6.1 - Verificação de quantas transações positivas há em cada fold

In [6]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

diagnostico_folds = []

for fold, (idx_treino, idx_val) in enumerate(
    tscv.split(X_cv),
    start=1
):

    diagnostico_folds.append({
        "Fold": fold,
        "Registros_treino": len(idx_treino),
        "Registros_validacao": len(idx_val),
        "Positivas_treino": int(y_cv.iloc[idx_treino].sum()),
        "Positivas_validacao": int(y_cv.iloc[idx_val].sum())
    })

diagnostico_folds = pd.DataFrame(diagnostico_folds)

display(diagnostico_folds)

if (diagnostico_folds["Positivas_treino"] == 0).any():
    raise ValueError("Existe fold sem exemplos positivos no treinamento.")

if (diagnostico_folds["Positivas_validacao"] == 0).any():
    raise ValueError("Existe fold sem exemplos positivos na validação.")

,Fold,Registros_treino,Registros_validacao,Positivas_treino,Positivas_validacao
0,1,33335,33333,50,31
1,2,66668,33333,81,34
2,3,100001,33333,115,38
3,4,133334,33333,153,41
4,5,166667,33333,194,33


### 6.2 - Função de Validação

In [7]:
def calcular_metricas(y_real, y_pred, y_prob):

    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(
            y_real, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_real, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_real, y_pred, zero_division=0
        ),
        "pr_auc": average_precision_score(
            y_real, y_prob
        )
    }


def validar_modelos_temporal(
    modelos,
    X,
    y,
    cv,
    limiar=0.5
):

    linhas = []

    for nome, modelo in modelos.items():

        inicio = time.perf_counter()

        print(f"\nAvaliando: {nome}")

        for fold, (tr, va) in enumerate(
            cv.split(X),
            start=1
        ):

            # Dados de treinamento do fold
            X_fold_train = X.iloc[tr]
            y_fold_train = y.iloc[tr]

            # Dados de validação do fold
            X_fold_val = X.iloc[va]
            y_fold_val = y.iloc[va]

            # Cópia independente do modelo
            modelo_fold = clone(modelo)

            # Treinamento
            modelo_fold.fit(
                X_fold_train,
                y_fold_train
            )

            # Probabilidades da classe positiva
            probabilidades = modelo_fold.predict_proba(
                X_fold_val
            )[:, 1]

            # Previsões com threshold padrão
            previsoes = (
                probabilidades >= limiar
            ).astype(int)

            # Métricas
            metricas = calcular_metricas(
                y_fold_val,
                previsoes,
                probabilidades
            )

            linhas.append({
                "modelo": nome,
                "fold": fold,
                "registros_treino": len(X_fold_train),
                "registros_validacao": len(X_fold_val),
                "positivas_treino": int(y_fold_train.sum()),
                "positivas_fold": int(y_fold_val.sum()),
                **metricas
            })

            print(
                f"Fold {fold} | "
                f"AP: {metricas['pr_auc']:.4f} | "
                f"Precision: {metricas['precision']:.4f} | "
                f"Recall: {metricas['recall']:.4f} | "
                f"F1: {metricas['f1']:.4f}"
            )

        print(
            f"Tempo total: {time.perf_counter() - inicio:.1f}s"
        )

    return pd.DataFrame(linhas)

In [8]:
cv_df = validar_modelos_temporal(
    modelos,
    X_cv,
    y_cv,
    tscv
)

display(cv_df)


Avaliando: Dummy
Fold 1 | AP: 0.0009 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Fold 2 | AP: 0.0010 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Fold 3 | AP: 0.0011 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Fold 4 | AP: 0.0012 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Fold 5 | AP: 0.0010 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Tempo total: 0.4s

Avaliando: LogisticRegression
Fold 1 | AP: 0.0742 | Precision: 0.0277 | Recall: 0.5161 | F1: 0.0526
Fold 2 | AP: 0.0577 | Precision: 0.0127 | Recall: 0.8235 | F1: 0.0249
Fold 3 | AP: 0.0466 | Precision: 0.0082 | Recall: 0.9737 | F1: 0.0164
Fold 4 | AP: 0.0478 | Precision: 0.0119 | Recall: 0.9268 | F1: 0.0234
Fold 5 | AP: 0.0965 | Precision: 0.0122 | Recall: 0.7879 | F1: 0.0240
Tempo total: 27.3s

Avaliando: RandomForest
Fold 1 | AP: 0.9264 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Fold 2 | AP: 0.7994 | Precision: 1.0000 | Recall: 0.0882 | F1: 0.1622
Fold 3 | AP: 0.7689 | Precision: 1.0000 | Recall:

,modelo,fold,registros_treino,registros_validacao,positivas_treino,positivas_fold,accuracy,precision,recall,f1,pr_auc
0,Dummy,1,33335,33333,50,31,0.999070,0.000000,0.000000,0.000000,0.000930
1,Dummy,2,66668,33333,81,34,0.998980,0.000000,0.000000,0.000000,0.001020
2,Dummy,3,100001,33333,115,38,0.998860,0.000000,0.000000,0.000000,0.001140
3,Dummy,4,133334,33333,153,41,0.998770,0.000000,0.000000,0.000000,0.001230
4,Dummy,5,166667,33333,194,33,0.999010,0.000000,0.000000,0.000000,0.000990
5,LogisticRegression,1,33335,33333,50,31,0.982720,0.027730,0.516129,0.052632,0.074202
6,LogisticRegression,2,66668,33333,81,34,0.934329,0.012664,0.823529,0.024944,0.057733
7,LogisticRegression,3,100001,33333,115,38,0.866529,0.008250,0.973684,0.016361,0.046647
8,LogisticRegression,4,133334,33333,153,41,0.904899,0.011856,0.926829,0.023413,0.047801
9,LogisticRegression,5,166667,33333,194,33,0.936639,0.012201,0.787879,0.024030,0.096549


# 7. Resumo dos Resultados

In [9]:
def resumo_cv(cv_df):

    agregacao = (
        cv_df
        .drop(columns=["fold", "positivas_fold"])
        .groupby("modelo")
        .agg(["mean", "std"])
    )

    agregacao.columns = [
        f"{metrica}_{estatistica}"
        for metrica, estatistica in agregacao.columns
    ]

    return agregacao.sort_values(
        "pr_auc_mean",
        ascending=False
    )


resumo = resumo_cv(cv_df)

display(
    resumo[
        [
            "precision_mean",
            "recall_mean",
            "f1_mean",
            "pr_auc_mean",
            "pr_auc_std"
        ]
    ]
)

,precision_mean,recall_mean,f1_mean,pr_auc_mean,pr_auc_std
modelo,,,,,
LightGBM,0.992857,0.817097,0.895742,0.942654,0.035963
RandomForest,0.800000,0.176767,0.281001,0.798563,0.074985
LogisticRegression,0.014540,0.805610,0.028276,0.064586,0.021004
Dummy,0.000000,0.000000,0.000000,0.001062,0.000121


# 8. Seleção do Vencedor

In [10]:
selecao_modelos = (
    cv_df
    .groupby("modelo")
    .agg(
        pr_auc_medio=("pr_auc", "mean"),
        pr_auc_desvio=("pr_auc", "std"),
        f1_medio=("f1", "mean"),
        precision_media=("precision", "mean"),
        recall_medio=("recall", "mean")
    )
    .sort_values(
        "pr_auc_medio",
        ascending=False
    )
)

display(selecao_modelos)

# Seleção do vencedor baseada exclusivamente na CV
melhor = selecao_modelos.index[0]

print(f"Modelo selecionado pela CV temporal: {melhor}")

,pr_auc_medio,pr_auc_desvio,f1_medio,precision_media,recall_medio
modelo,,,,,
LightGBM,0.942654,0.035963,0.895742,0.992857,0.817097
RandomForest,0.798563,0.074985,0.281001,0.800000,0.176767
LogisticRegression,0.064586,0.021004,0.028276,0.014540,0.805610
Dummy,0.001062,0.000121,0.000000,0.000000,0.000000


Modelo selecionado pela CV temporal: LightGBM


# 9. Treinamento Final do Vencedor

In [11]:
# Amostra de treinamento para o modelo final
if MAX_AMOSTRA_TREINO is None or len(X_train) <= MAX_AMOSTRA_TREINO:

    X_train_execucao = X_train
    y_train_execucao = y_train

else:

    X_train_execucao = X_train.sample(
        n=MAX_AMOSTRA_TREINO,
        random_state=SEMENTE
    ).sort_index()

    y_train_execucao = y_train.loc[
        X_train_execucao.index
    ]

# Cria uma cópia independente do vencedor
modelo_vencedor = clone(modelos[melhor])

# Treina o vencedor
inicio = time.perf_counter()

modelo_vencedor.fit(
    X_train_execucao,
    y_train_execucao
)

tempo_treino = time.perf_counter() - inicio

print(f"Modelo treinado: {melhor}")
print(f"Tempo de treinamento: {tempo_treino:.1f}s")
print(f"Registros utilizados: {len(X_train_execucao)}")

Modelo treinado: LightGBM
Tempo de treinamento: 7.2s
Registros utilizados: 500000


# 10. Escolha do Threshold na Validação

In [12]:
# Probabilidades na validação
probabilidades_val = modelo_vencedor.predict_proba(
    X_val
)[:, 1]

# Curva Precision-Recall
precisions, recalls, thresholds = precision_recall_curve(
    y_val,
    probabilidades_val
)

# Calcula F1 para cada threshold
f1_scores = (
    2 * precisions[:-1] * recalls[:-1]
    / np.clip(
        precisions[:-1] + recalls[:-1],
        1e-12,
        None
    )
)

# Melhor threshold
idx_melhor = np.argmax(f1_scores)

threshold_otimo = thresholds[idx_melhor]

print(f"Threshold selecionado: {threshold_otimo:.4f}")
print(f"F1 na validação: {f1_scores[idx_melhor]:.4f}")

Threshold selecionado: 0.9339
F1 na validação: 0.8739


# 11. Avaliação Final no Teste

In [13]:
# Probabilidades no teste
probabilidades_teste = modelo_vencedor.predict_proba(
    X_test
)[:, 1]

# Classificação utilizando o threshold escolhido na validação
previsoes_teste = (
    probabilidades_teste >= threshold_otimo
).astype(int)

# Métricas finais
resultados_finais = {
    "Modelo": melhor,
    "Threshold": threshold_otimo,

    "Accuracy": accuracy_score(
        y_test,
        previsoes_teste
    ),

    "Precision": precision_score(
        y_test,
        previsoes_teste,
        zero_division=0
    ),

    "Recall": recall_score(
        y_test,
        previsoes_teste,
        zero_division=0
    ),

    "F1": f1_score(
        y_test,
        previsoes_teste,
        zero_division=0
    ),

    "PR-AUC": average_precision_score(
        y_test,
        probabilidades_teste
    )
}

resultado_teste = pd.DataFrame([resultados_finais])

display(resultado_teste.round(4))

,Modelo,Threshold,Accuracy,Precision,Recall,F1,PR-AUC
0,LightGBM,0.9339,0.9997,0.8983,0.8433,0.8699,0.92
